# Sentiment Analysis
**Project A - Group 10**

Notebook ini bertujuan untuk menganalisis sentimen artikel berita menggunakan machine learning.

Dataset: `scraped_articles_preprocessed.csv` (804 artikel)  
Label sentimen: **Positif**, **Negatif**, **Netral**

---

## 1. Import Library

In [ ]:
import pandas as pd
import ast
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score

print('Library berhasil diimport.')

## 2. Load Dataset

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/MirzaFathi/Project-A---10/main/Data/scraped_articles_preprocessed.csv'

df = pd.read_csv(DATA_URL)

print('Jumlah data:', len(df))
print('Kolom:', df.columns.tolist())
df.head(3)

## 3. Eksplorasi Data

Cek distribusi label sentimen untuk memastikan data tidak terlalu tidak seimbang.

In [ ]:
print('Distribusi Sentimen:')
print(df['sentimen'].value_counts())

In [ ]:
sentimen_counts = df['sentimen'].value_counts()

plt.figure(figsize=(6, 4))
plt.bar(sentimen_counts.index, sentimen_counts.values, color=['#4CAF50', '#F44336', '#2196F3'])
plt.title('Distribusi Label Sentimen')
plt.xlabel('Sentimen')
plt.ylabel('Jumlah Artikel')

for i, v in enumerate(sentimen_counts.values):
    plt.text(i, v + 2, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Persiapan Data

Kolom `stemming_cleaned` berisi token hasil preprocessing dalam bentuk list.  
Kita ubah menjadi string agar bisa diproses oleh TF-IDF.

In [ ]:
def tokens_to_string(val):
    try:
        tokens = ast.literal_eval(val)
        return ' '.join(tokens)
    except:
        return str(val)

df['teks'] = df['stemming_cleaned'].apply(tokens_to_string)

label_map = {'Negatif': 0, 'Netral': 1, 'Positif': 2}
df['label'] = df['sentimen'].map(label_map)

print('Contoh teks:')
print(df['teks'].iloc[0][:200])
print()
print('Label mapping:', label_map)

## 5. Split Data (Training & Testing)

Data dibagi 80% untuk training dan 20% untuk testing.  
Parameter `stratify=y` memastikan proporsi tiap kelas sama di kedua set.

In [ ]:
X = df['teks']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Data training : {len(X_train)} artikel')
print(f'Data testing  : {len(X_test)} artikel')

## 6. TF-IDF Vectorization

TF-IDF mengubah teks menjadi angka. Kata yang sering muncul di satu dokumen tapi jarang di dokumen lain akan mendapat bobot tinggi.

In [ ]:
tfidf = TfidfVectorizer(max_features=3000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'Jumlah fitur TF-IDF: {X_train_tfidf.shape[1]}')

## 7. Model 1 - Naive Bayes

Naive Bayes adalah model klasik untuk klasifikasi teks. Model ini bekerja dengan menghitung probabilitas setiap kata muncul di tiap kelas sentimen.

In [ ]:
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

y_pred_nb = nb_model.predict(X_test_tfidf)

print(f'Akurasi Naive Bayes: {accuracy_score(y_test, y_pred_nb):.2%}')

In [ ]:
label_names = ['Negatif', 'Netral', 'Positif']

print('Classification Report - Naive Bayes')
print('=' * 50)
print(classification_report(y_test, y_pred_nb, target_names=label_names))

In [ ]:
cm_nb = confusion_matrix(y_test, y_pred_nb)

plt.figure(figsize=(6, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_nb, display_labels=label_names)
disp.plot(colorbar=False, cmap='Oranges')
plt.title('Confusion Matrix - Naive Bayes')
plt.tight_layout()
plt.show()

## 8. Model 2 - Logistic Regression

Logistic Regression adalah model yang belajar mencari batas pemisah antar kelas. Model ini sangat cocok digunakan bersama TF-IDF karena mampu menangani banyak fitur sekaligus.

In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)

y_pred_lr = lr_model.predict(X_test_tfidf)

print(f'Akurasi Logistic Regression: {accuracy_score(y_test, y_pred_lr):.2%}')

In [ ]:
print('Classification Report - Logistic Regression')
print('=' * 50)
print(classification_report(y_test, y_pred_lr, target_names=label_names))

In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)

plt.figure(figsize=(6, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=label_names)
disp.plot(colorbar=False, cmap='Blues')
plt.title('Confusion Matrix - Logistic Regression')
plt.tight_layout()
plt.show()

## 9. Perbandingan Hasil

Bandingkan akurasi kedua model secara visual.

In [ ]:
acc_nb = accuracy_score(y_test, y_pred_nb)
acc_lr = accuracy_score(y_test, y_pred_lr)

models = ['Naive Bayes', 'Logistic Regression']
scores = [acc_nb, acc_lr]

plt.figure(figsize=(6, 4))
bars = plt.bar(models, scores, color=['coral', 'steelblue'])
plt.ylim(0, 1.1)
plt.title('Perbandingan Akurasi Model')
plt.ylabel('Akurasi')

for bar, score in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{score:.2%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Naive Bayes         : {acc_nb:.2%}')
print(f'Logistic Regression : {acc_lr:.2%}')

## 10. Kesimpulan

Dua model digunakan untuk memprediksi sentimen artikel (Positif / Netral / Negatif):

- **Naive Bayes**: model sederhana dan cepat, cocok sebagai baseline klasifikasi teks
- **Logistic Regression**: model yang lebih fleksibel, umumnya menghasilkan akurasi lebih tinggi

Berdasarkan hasil di atas, **Logistic Regression** menjadi model utama karena performanya lebih baik pada dataset ini.